In [8]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
from langchain_community.document_loaders import GitLoader
loader = GitLoader(
    clone_url="https://github.com/viswa1220/TossTheTurf",
    repo_path="Test",
    branch="owner",
    file_filter=lambda file_path: file_path.endswith(
        (".js", ".py", ".md", ".json", ".jsx", ".tsx")
    ) and "node_modules" not in file_path
    and "package-lock" not in file_path
    and "build/" not in file_path
)
docs = loader.load()
print(f"Loaded {len(docs)} files")

Loaded 55 files


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

Created 277 chunks


In [7]:
from langchain_chroma import Chroma
from langchain_community.embeddings import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="gemma:2b")
vectorDb = Chroma.from_documents(chunks, embedding=embeddings, persist_directory="./chroma_db_v2")
print("Done. Saved to disk.")

Done. Saved to disk.


In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o")
retriever = vectorDb.as_retriever()
query="how to setup frontend of this codebase"
relevant_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in relevant_docs])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a developer onboarding assistant. Answer questions about the codebase using the context below.\n\nContext:\n{context}"),
    ("human", "{input}")
])

chain = prompt | llm
response = chain.invoke({"context": context, "input": query})
print(response.content)

To set up the frontend of this codebase, which is based on Create React App, follow these steps:

1. **Clone the Repository**:
   If you haven't already, clone the code repository to your local machine using Git.

   ```sh
   git clone <repository-url>
   ```

2. **Navigate to the Project Directory**:
   Change into the directory of the newly cloned repository.

   ```sh
   cd <project-directory>
   ```

3. **Install Dependencies**:
   Use npm to install the required dependencies for the project.

   ```sh
   npm install
   ```

4. **Run the Development Server**:
   Start the application in development mode. This will open a browser window pointing to the local server where your app is running.

   ```sh
   npm start
   ```

   - Open [http://localhost:3000](http://localhost:3000) in your browser to view the app.
   - The page will automatically reload when you make changes to the source files.
   - You may also see any lint errors in the console, which you can fix accordingly.

Follow